# Phase VII — Full ArtBench-10 confirmatory analysis (diagnostic logging)

Use this version if the previous launcher stopped with only a generic `exit code 1`. It streams **all stdout + stderr** from the Phase VII subprocess to the notebook and simultaneously writes a persistent log to Google Drive. On failure it automatically prints the last 100 log lines.

Run **Runtime → Run all**. Existing Drive checkpoints are preserved and reused.


In [ ]:
# 0. Setup
import os, sys, subprocess, shutil
from pathlib import Path
from google.colab import drive, files

drive.mount('/content/drive')
REPO_URL='https://github.com/ardominguezm/painting-geometry.git'
BRANCH='multiscale-corpus-analysis'
REPO_DIR=Path('/content/painting-geometry')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
clone=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],text=True,capture_output=True)
if clone.returncode!=0:
    print(clone.stdout); print(clone.stderr); raise RuntimeError('Git clone failed')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO_DIR/'requirements.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ordpy>=1.2.0','kagglehub','requests'],check=True)
commit=subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
print('Repository cloned ✓'); print('Branch:',BRANCH); print('Commit:',commit)


## 1. Preflight imports

This detects dependency/import errors before starting the long job.


In [ ]:
preflight='''import sys
from pathlib import Path
repo=Path("/content/painting-geometry")
sys.path.insert(0,str(repo))
import numpy,pandas,scipy,skimage,sklearn,cv2,PIL,requests,tqdm
from src.baselines import lbp_features,multidistance_glcm_features,multiscale_gradient_features,orientation_histogram_features
from src.curvature_v2 import relative_scale_curvature_features
from src.orientation import structure_tensor_features
from src.preprocessing import preprocess
print("PREFLIGHT IMPORTS OK")
'''
p=subprocess.run([sys.executable,'-c',preflight],text=True,capture_output=True)
print(p.stdout)
if p.returncode!=0:
    print(p.stderr)
    raise RuntimeError('Preflight import failed; traceback printed above.')


## 2. Run complete Phase VII with persistent log

The log is saved at `MyDrive/painting_geometry_phase7_full/phase7_last_run.log`. On any error, the last 100 lines are printed automatically.


In [ ]:
DRIVE_ROOT=Path('/content/drive/MyDrive/painting_geometry_phase7_full')
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
LOG=DRIVE_ROOT/'phase7_last_run.log'
cmd=[sys.executable,'-u',str(REPO_DIR/'scripts'/'run_phase7_full_pipeline.py'),'--repo-dir',str(REPO_DIR),'--drive-root',str(DRIVE_ROOT),'--feature-chunk-size','500','--ordinal-checkpoint-every','5000','--n-permutations','4999','--n-bootstrap','5000']
print('Starting Phase VII...'); print('Persistent root:',DRIVE_ROOT); print('Log:',LOG)
with open(LOG,'w',encoding='utf-8') as log:
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end='')
        log.write(line); log.flush()
    rc=proc.wait()
if rc!=0:
    lines=LOG.read_text(encoding='utf-8',errors='replace').splitlines()
    print('\n'+'='*80); print('LAST 100 LINES OF PHASE VII LOG'); print('='*80)
    print('\n'.join(lines[-100:]))
    raise RuntimeError(f'Phase VII stopped with exit code {rc}. The exact failing traceback is printed above and saved in {LOG}. Existing checkpoints are preserved.')
print('Phase VII complete ✓')


## 3. Download compact results


In [ ]:
LIGHT=DRIVE_ROOT/'painting_geometry_phase7_full_results_LIGHT.zip'
FEATURES_ZIP=DRIVE_ROOT/'painting_geometry_phase7_feature_matrices.zip'
if not LIGHT.exists(): raise FileNotFoundError(LIGHT)
print('Compact results:',LIGHT,f'{LIGHT.stat().st_size/1e6:.1f} MB')
if FEATURES_ZIP.exists(): print('Full feature archive in Drive:',FEATURES_ZIP,f'{FEATURES_ZIP.stat().st_size/1e6:.1f} MB')
files.download(str(LIGHT))
